# Explore Cerner sandbox FHIR data

Run cells top to bottom. Part 1 needs no login. Part 2 uses your app secret.

In [ ]:
import os, json, requests

TENANT = "ec2458f2-1e24-41c8-b71b-0e701af7583d"  # Cerner public sandbox tenant
HEADERS = {"Accept": "application/fhir+json"}

def show(data):
    print(json.dumps(data, indent=2)[:3000])  # first 3000 chars is enough to see the shape

## Part 1 - Open sandbox (no auth, fake patients)

In [ ]:
OPEN_URL = f"https://fhir-open.cerner.com/r4/{TENANT}"

r = requests.get(f"{OPEN_URL}/Patient", params={"family": "SMART", "_count": 5}, headers=HEADERS)
bundle = r.json()
show(bundle)

In [ ]:
# Pick the first patient and look at the data a transfer summary would use
patient_id = bundle["entry"][0]["resource"]["id"]
print("Patient:", patient_id)

for resource in ["Condition", "Encounter", "Observation", "MedicationRequest", "AllergyIntolerance"]:
    r = requests.get(f"{OPEN_URL}/{resource}", params={"patient": patient_id, "_count": 3}, headers=HEADERS)
    print(f"\n===== {resource} ({r.status_code}) =====")
    show(r.json())

In [ ]:
# Bed info lives in Location resources
r = requests.get(f"{OPEN_URL}/Location", params={"_count": 5}, headers=HEADERS)
show(r.json())

## Part 2 - Your app (DischargeAgent) with the secret

Set the secret as an environment variable before starting Jupyter, never paste it in the notebook:

```
export CERNER_SECRET=...
```

In [ ]:
CLIENT_ID = "4a81c10d-fd9c-433f-8a01-3161ba48d631"  # system account ID
SECRET = os.environ["CERNER_SECRET"]

TOKEN_URL = f"https://authorization.cerner.com/tenants/{TENANT}/protocols/oauth2/profiles/smart-v1/token"
SECURE_URL = f"https://fhir-ehr-code.cerner.com/r4/{TENANT}"

r = requests.post(
    TOKEN_URL,
    auth=(CLIENT_ID, SECRET),
    data={"grant_type": "client_credentials", "scope": "system/Patient.read system/Encounter.read system/Condition.read"},
)
print(r.status_code)
show(r.json())
token = r.json().get("access_token")

In [ ]:
auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
r = requests.get(f"{SECURE_URL}/Patient/{patient_id}", headers=auth_headers)
print(r.status_code)
show(r.json())